# Orquestrador Contínuo: Bronze ➔ Silver
Notebook mestre responsável pelo loop contínuo de polling e execução em cadeia dos pipelines.

In [0]:
import json
import time

poll_interval_seconds = 15
caminho_bronze = "04_ingestao_bronze"
caminho_silver = "05_ingestao_silver"

# Configuração da flag de parada por inatividade
MAX_VERIFICACOES_SEM_DADOS = 3
verificacoes_sem_dados = 0

print("Iniciando orquestração inteligente Bronze ➔ Silver...\n")

try:
    while True:
        agora = time.strftime('%Y-%m-%d %H:%M:%S')
        print(f"[{agora}] ─── Verificando novos dados ───")

        # 1. Executa a Bronze e recebe o relatório de ingestão
        try:
            retorno_raw = dbutils.notebook.run(caminho_bronze, timeout_seconds=1200)
            res_bronze = json.loads(retorno_raw)
            qtd = res_bronze.get("qtd_ingeridos", 0)
            arquivos = res_bronze.get("arquivos", [])

            if qtd > 0:
                print(f"  ✔ Bronze ingeriu {qtd} novo(s) arquivo(s):")
                for arq in arquivos:
                    print(f"     └─ {arq}")
            else:
                print("  ℹ Bronze: Nenhum arquivo novo encontrado.")

        except Exception as e:
            print(f"  ✖ Erro na execução da Bronze: {e}")
            qtd = 0

        # 2. Decide se executa a Silver e atualiza o contador de inatividade
        if qtd > 0:
            verificacoes_sem_dados = 0  # Reseta o contador se encontrou dados novos
            print("  ▶ Iniciando Silver para processar os novos micro-lotes...")
            try:
                dbutils.notebook.run(caminho_silver, timeout_seconds=1200)
                print("  ✔ Silver finalizada com sucesso.")
            except Exception as e:
                print(f"  ✖ Erro na execução da Silver: {e}")
        else:
            verificacoes_sem_dados += 1
            print(f"  ⏭ Silver ignorada: Não há novos dados para processar. (Sem novos dados: {verificacoes_sem_dados}/{MAX_VERIFICACOES_SEM_DADOS})")

        # 3. Interrupção do loop após 3 verificações sem novos dados
        if verificacoes_sem_dados >= MAX_VERIFICACOES_SEM_DADOS:
            print(f"\n⏹ Limite de {MAX_VERIFICACOES_SEM_DADOS} verificações consecutivas sem novos dados atingido. Encerrando orquestrador.")
            break

        print(f"─── Aguardando {poll_interval_seconds}s para a próxima checagem ───\n")
        time.sleep(poll_interval_seconds)

except KeyboardInterrupt:
    print("\nOrquestrador interrompido manualmente pelo usuário.")